In [1]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

Se consideran las funciones

$$k(x) = x^4 - 2x^3 - 2x, \quad h(x) = \cos(x) - 1.$$

a) Encontrar el punto de intersección de ambas curvas usando el método de Traub con $x_0 = 3$, y una tolerancia de $10^{-12}$, indicar el valor del error, el número de iteraciones y el ACOC que se obtiene. Comparar los resultados obtenidos con el Método SH4 y el Método MMC2 (En Python)

In [2]:
def TraubACOC(f,df,x0,tol,maxiter):
    iters = 0
    error = tol+1
    error_x = list()

    while error>tol and iters<=maxiter:

        y = x0-f(x0)/df(x0)
        x = y-f(y)/df(x0)
        error = math.fabs(x-x0)
        error_x.append(error)
        iters = iters+1
        x0 = x

    error_x = np.array(error_x)
    ACOC = np.log(error_x[2:]/error_x[1:-1])/np.log(error_x[1:-1]/error_x[0:-2])
    
    return [x,iters,error,ACOC]

def SH4(f,df,x0,tol,maxiter=30):
    iters = 0
    error = tol + 1
    error_list = list()

    while error > tol and iters <= maxiter:
        y = x0 - (f(x0)/df(x0))
        z = y - f(x0)/(f(x0)-2*f(y)) * (f(y)/df(x0))
        x = z - (f(x0)+2*f(y))/(f(x0)+4*f(y)) * (f(z)/df(x0))

        error = math.fabs(x-x0)
        error_list.append(error)
        x0 = x
        iters += 1
        
    error_list = np.array(error_list)
    ACOC = np.log(error_list[2:]/error_list[1:-1])/np.log(error_list[1:-1]/error_list[0:-2])
    
    return [x,iters,error,ACOC]

def MMC2(f,df,x0,tol,maxiter=30):
    iters = 0
    error = tol + 1
    error_list = list()

    while error > tol and iters <= maxiter:
        y = x0 - (f(x0)/df(x0))
        z = y - f(x0)/(f(x0)-2*f(y)) * (f(y)/df(x0))
        if f(x0)-f(y) == 0:
            # x0 ya es raíz a precisión de máquina: f(x0) == f(y) por redondeo
            x = z
        else:
            x = z - (2 + 3*f(y)/f(x0) - f(x0)/(f(x0)-f(y))) * (f(z)/df(x0))

        error = math.fabs(x-x0)
        error_list.append(error)
        x0 = x
        iters += 1

    error_list = np.array(error_list)
    ACOC = np.log(error_list[2:]/error_list[1:-1])/np.log(error_list[1:-1]/error_list[0:-2])

    return [x,iters,error,ACOC]

In [3]:
# Intersección: k(x) = h(x)  =>  f(x) = k(x) - h(x) = 0
f = lambda x: x**4 - 2*x**3 - 2*x - math.cos(x) + 1
df = lambda x: 4*x**3 - 6*x**2 - 2 + math.sin(x)

x0 = 3
tol = 1e-12
maxiter = 50

resultados = {
    'Traub': TraubACOC(f,df,x0,tol,maxiter),
    'SH4': SH4(f,df,x0,tol,maxiter),
    'MMC2': MMC2(f,df,x0,tol,maxiter),
}

for metodo, r in resultados.items():
    print(metodo, 'ACOC por iteración:', r[3])

# El último error es 0 exacto, por eso el último ACOC sale inf; se toma el último valor finito
tabla = pd.DataFrame({
    'Raíz': [r[0] for r in resultados.values()],
    'Iteraciones': [r[1] for r in resultados.values()],
    'Error': [r[2] for r in resultados.values()],
    'ACOC': [r[3][np.isfinite(r[3])][-1] for r in resultados.values()],
}, index=resultados.keys())

pd.set_option('display.precision', 15)
print("\n Tabal comparativa de los resultados: ")
tabla

Traub ACOC por iteración: [2.53576625 2.8503908         inf]
SH4 ACOC por iteración: [4.05357514        inf]
MMC2 ACOC por iteración: [4.87878428        inf]

 Tabal comparativa de los resultados: 


/var/folders/qt/89mhcy710gj3n350ps35v5600000gn/T/ipykernel_15498/617912874.py:16: RuntimeWarning: divide by zero encountered in log
  ACOC = np.log(error_x[2:]/error_x[1:-1])/np.log(error_x[1:-1]/error_x[0:-2])
/var/folders/qt/89mhcy710gj3n350ps35v5600000gn/T/ipykernel_15498/617912874.py:36: RuntimeWarning: divide by zero encountered in log
  ACOC = np.log(error_list[2:]/error_list[1:-1])/np.log(error_list[1:-1]/error_list[0:-2])
/var/folders/qt/89mhcy710gj3n350ps35v5600000gn/T/ipykernel_15498/617912874.py:60: RuntimeWarning: divide by zero encountered in log
  ACOC = np.log(error_list[2:]/error_list[1:-1])/np.log(error_list[1:-1]/error_list[0:-2])


,Raíz,Iteraciones,Error,ACOC
Traub,2.251727153575569,5,0.0,2.850390802866477
SH4,2.251727153575569,4,0.0,4.053575140063467
MMC2,2.251727153575569,4,0.0,4.878784279124116


### Resumen de la comparación

Los tres métodos llegan a la misma raíz $a \approx 2.251727153575569$. **Traub** necesita 5 iteraciones (ACOC ≈ 2.85, orden 3), mientras que **SH4** (ACOC ≈ 4.05) y **MMC2** (ACOC ≈ 4.88) necesitan solo 4. **MMC2** es el más eficiente: tiene el mismo costo por iteración que SH4, pero converge más rápido.

b) Con el resultado anterior, encontrar el valor de la integral (Integrar en papel y lápiz)

$$\int_{0}^{a} (h(x) - k(x)) \, dx$$

$$k(x) = x^4 - 2x^3 - 2x, \quad h(x) = \cos(x) - 1.$$